# Gold results – NYC 311 + PLUTO

Reads the finished **Gold** tables written by `gold_combined.py` and answers the five project questions.

Same way of presenting results as the labs: `.show()` tables and `spark.sql()` on temp views, plus one chart per question for the report.

Gold tables are small aggregates, so this notebook is light: it does **not** re-read Silver and needs no cache. Charts are saved to `results/` (next to `notebooks/`).

## 0. Setup

In [ ]:
import findspark
findspark.init()

import os
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

GOLD = "/home/ubuntu/spark-project-data/dev/gold"
RESULTS_DIR = "../results"
os.makedirs(RESULTS_DIR, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("DAT535-Gold-Results")
    .master("spark://192.168.18.168:7077")
    .config("spark.driver.host", "192.168.18.168")
    .config("spark.cores.max", "4")          # small job: only reads Gold
    .config("spark.executor.memory", "2g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

# PLUTO land-use codes -> readable names
LAND_USE = {
    "1": "One & two family", "2": "Multi-family walk-up", "3": "Multi-family elevator",
    "4": "Mixed res. & commercial", "5": "Commercial & office", "6": "Industrial & manufacturing",
    "7": "Transportation & utility", "8": "Public facilities", "9": "Open space & recreation",
    "10": "Parking", "11": "Vacant land",
}
# Create a Spark SQL map for land-use codes -> names
land_use_name = F.create_map(*[x for k, v in LAND_USE.items() for x in (F.lit(k), F.lit(v))])


def save_bar(pdf, label_col, value_col, title, xlabel, filename):
    """Horizontal bar chart from a small pandas DataFrame; saved for the report."""
    pdf = pdf.iloc[::-1]                       # largest on top
    fig, ax = plt.subplots(figsize=(9, 0.4 * len(pdf) + 1.5))
    ax.barh(pdf[label_col].astype(str), pdf[value_col])
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    plt.tight_layout()
    fig.savefig(f"{RESULTS_DIR}/{filename}", dpi=150)
    plt.show()

## 1. Load Gold tables and check them (like Lab 2 `generate_summary`)

In [ ]:
TABLES = [
    # Member A
    "demand_by_area_month", "demand_by_complaint_borough",
    "resolution_by_agency_complaint", "resolution_by_borough_month",
    # Member B
    "property_profile", "property_repeat_problems", "property_repeat_summary",
    "property_peer_comparison", "property_type_variety",
]

gold = {}
for name in TABLES:
    gold[name] = spark.read.parquet(f"{GOLD}/{name}")
    gold[name].createOrReplaceTempView(name)     # for spark.sql() below (Lab 2 style)
    print(f"{name:<35} {gold[name].count():>10,} rows")

---
## Q1 – Where is service demand concentrated?

In [ ]:
print("=== Requests per borough ===")
spark.sql("""
    SELECT borough, SUM(total_requests) AS requests
    FROM demand_by_area_month
    GROUP BY borough
    ORDER BY requests DESC
""").show()

print("=== Top 10 community boards ===")
top_boards = spark.sql("""
    SELECT community_board, borough, SUM(total_requests) AS requests
    FROM demand_by_area_month
    WHERE community_board IS NOT NULL
    GROUP BY community_board, borough
    ORDER BY requests DESC
    LIMIT 10
""")
top_boards.show(truncate=False)

print("=== Top 3 complaint types in each borough ===")
gold["demand_by_complaint_borough"] \
    .filter("rank_in_borough <= 3") \
    .orderBy("borough", "rank_in_borough") \
    .show(20, truncate=False)

In [ ]:
save_bar(top_boards.toPandas(), "community_board", "requests",
         "Q1: Top 10 community boards by number of 311 requests",
         "Requests", "q1_top_community_boards.png")

---
## Q2 – Where is resolution slow or inconsistent?
Slow = high median / p90 hours. Inconsistent = high `cv_hours` (stddev ÷ mean). Only groups with `enough_data`.

In [ ]:
res = gold["resolution_by_agency_complaint"].filter("enough_data")

print("=== 10 slowest agency + complaint type (median hours) ===")
slowest = res.orderBy(F.desc("median_hours")) \
    .select("agency", "complaint_type", "closed_with_valid_time",
            "median_hours", "p90_hours", "pct_unresolved").limit(10)
slowest.show(truncate=False)

print("=== 10 most inconsistent (highest cv_hours) ===")
res.orderBy(F.desc("cv_hours")) \
    .select("agency", "complaint_type", "closed_with_valid_time",
            "avg_hours", "stddev_hours", "cv_hours").limit(10) \
    .show(truncate=False)

print("=== Median resolution hours per borough (all months) ===")
spark.sql("""
    SELECT borough,
           ROUND(AVG(median_hours), 1) AS avg_monthly_median_hours,
           ROUND(AVG(cv_hours), 2)     AS avg_monthly_cv
    FROM resolution_by_borough_month
    WHERE enough_data
    GROUP BY borough
    ORDER BY avg_monthly_median_hours DESC
""").show()

In [ ]:
pdf = slowest.toPandas()
pdf["label"] = pdf["agency"] + " – " + pdf["complaint_type"]
save_bar(pdf, "label", "median_hours",
         "Q2: 10 slowest agency / complaint types (median hours to close)",
         "Median resolution hours", "q2_slowest_resolution.png")

---
## Q3 – Which properties have repeated service problems?
Repeated = same complaint type at the same property ≥ 3 times in ≥ 2 different months.

In [ ]:
repeat = gold["property_repeat_summary"]
total_props = gold["property_profile"].count()
repeat_props = repeat.count()
print(f"Properties with repeated problems: {repeat_props:,} of {total_props:,} "
      f"({repeat_props / total_props * 100:.1f}%)")

print("=== Top 10 properties by repeated requests ===")
repeat.withColumn("land_use_name", land_use_name[F.col("land_use")]) \
    .select("bbl", "borough", "land_use_name", "residential_units",
            "repeated_problem_types", "repeated_requests", "top_repeated_type", "repeat_share") \
    .orderBy(F.desc("repeated_requests")).show(10, truncate=False)

print("=== Most common repeated problem types ===")
top_repeat_types = spark.sql("""
    SELECT complaint_type, COUNT(*) AS properties
    FROM property_repeat_problems
    GROUP BY complaint_type
    ORDER BY properties DESC
    LIMIT 10
""")
top_repeat_types.show(truncate=False)

In [ ]:
save_bar(top_repeat_types.toPandas(), "complaint_type", "properties",
         "Q3: Complaint types that most often repeat at the same property",
         "Number of properties", "q3_repeated_problem_types.png")

---
## Q4 – Which properties have unusually high activity vs similar buildings?
Peer group = borough + land use + residential-unit size band. Flagged = above peer p90 and ≥ 3× peer median (groups with ≥ 10 properties).

In [ ]:
peer = gold["property_peer_comparison"] \
    .withColumn("land_use_name", land_use_name[F.col("peer_land_use")])

flagged = peer.filter("is_unusually_high")
n_flagged, n_all = flagged.count(), peer.count()
print(f"Unusually high properties: {n_flagged:,} of {n_all:,} ({n_flagged / n_all * 100:.2f}%)")

print("=== Top 10 by activity ratio (requests ÷ peer median) ===")
flagged.select("bbl", "peer_borough", "land_use_name", "unit_band", "total_requests",
               "peer_median_requests", "activity_ratio", "peer_percentile") \
    .orderBy(F.desc("activity_ratio")).show(10, truncate=False)

print("=== Flagged properties per land use ===")
flagged_by_use = flagged.groupBy("land_use_name").count().orderBy(F.desc("count"))
flagged_by_use.show(truncate=False)

In [ ]:
save_bar(flagged_by_use.toPandas().fillna("Unknown"), "land_use_name", "count",
         "Q4: Properties with unusually high complaint activity, by land use",
         "Flagged properties", "q4_flagged_by_land_use.png")

---
## Q5 – Which property types have the widest variety of service needs?
Only property types with ≥ 30 properties (`enough_properties`).

In [ ]:
variety = gold["property_type_variety"].filter("enough_properties") \
    .withColumn("land_use_name", land_use_name[F.col("land_use")])

print("=== Top 15 property types by complaint variety per property ===")
top_variety = variety.select("land_use_name", "building_class", "properties",
                             "avg_types_per_property", "avg_agencies_per_property",
                             "distinct_complaint_types") \
    .orderBy(F.desc("avg_types_per_property")).limit(15)
top_variety.show(truncate=False)

In [ ]:
pdf = top_variety.toPandas()
pdf["label"] = pdf["building_class"].astype(str) + " (" + pdf["land_use_name"].fillna("Unknown") + ")"
save_bar(pdf, "label", "avg_types_per_property",
         "Q5: Property types with the widest variety of service needs",
         "Average distinct complaint types per property", "q5_property_type_variety.png")

## Clean up

In [ ]:
spark.stop()